# Train

In [ ]:
# 1. Install the Ultralytics library
!pip install ultralytics -q

# 2. Install Roboflow and download your dataset
!pip install roboflow

!pip install python-dotenv

MODEL_VERSION = 1

from roboflow import Roboflow
from dotenv import load_dotenv
import os

load_dotenv()

rf = Roboflow(api_key=os.getenv("ROBOFLOW_API"))

project = rf.workspace("noams-workspace-vecda").project("superstrika")
version = project.version(MODEL_VERSION)
dataset = version.download("yolov8")


# 3. Train a lightweight Nano model
from ultralytics import YOLO
model = YOLO('yolov8n.pt')  # Starts with a pre-trained 'nano' model weights

import yaml
import os

yaml_path = f'/content/Superstrika-{MODEL_VERSION}/data.yaml'

# 1. Load your current data.yaml file
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

print("Original data.yaml:", data)
print("Actual folders found:", os.listdir(f'/content/Superstrika-{MODEL_VERSION}'))

# 2. Force the correct absolute root path
data['path'] = f'/content/Superstrika-{MODEL_VERSION}'

# 3. Automatically detect where your validation images are
if os.path.exists(f'/content/Superstrika-{MODEL_VERSION}/valid/images'):
    data['val'] = 'valid/images'
elif os.path.exists(f'/content/Superstrika-{MODEL_VERSION}/val/images'):
    data['val'] = 'val/images'
else:
    # Fallback: If Roboflow didn't make a validation split, use the train folder so it doesn't crash
    print("⚠️ No validation folder found! Routing validation to train folder.")
    data['val'] = 'train/images'

# 4. Save the corrected data.yaml back to disk
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("✅ data.yaml successfully updated!")

# Train on your custom data. 50 epochs is a great start for a simple ball.
model.train(data=f"/content/Superstrika-{MODEL_VERSION}/data.yaml", epochs=50, imgsz=640)

# Export

In [ ]:
TRAIN_RUN_NUMBER = 3

from ultralytics import YOLO

# Load your freshly trained model weights
model = YOLO(f'/content/runs/detect/train-{TRAIN_RUN_NUMBER}/weights/best.pt')

# Export to ONNX format (highly compatible and much faster on CPUs)
model.export(format='onnx', dynamic=True)